# Lab 6: Roofline Analysis for LLM Inference

<!-- The badge resolves against the PUBLIC mirror repo, which
     .github/workflows/sync-gpu-labs-student.yml keeps in step with
     gpu-labs/student/. Viewed inside the private instructor repo the image
     renders but the click-through 404s until that mirror has run. -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab6_Roofline_Analysis_STUDENT.ipynb)

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Week**: 6, Performance Modeling  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)  
**Runtime**: Google Colab, GPU runtime required (T4)  
**Expected time**: 90–120 minutes

---

## Learning objectives
By the end of this lab you will be able to:
1. Compute arithmetic intensity for common operators from first principles.
2. Construct an empirical roofline for your GPU from measured peaks.
3. Profile real LLM kernels and place them on the roofline.
4. Compare GPU and IBM Spyre with a dual roofline and a capacity-aware crossover.

## Prerequisites
Labs 1-5 complete, plus the Week 6 lecture on the roofline model, arithmetic intensity, and the ridge point.

## How this Self-Assessment Lab works

**Practice Lab**: You run this lab on **Google Colab** (free T4).  
**OPTIONAL SELF GRADING:** **A separate grader notebook is available on Coursera Labs for self-evaluation**.

| | Where | What happens |
|---|---|---|
| **1. This notebook** | Google Colab (T4 GPU) | Complete every `TODO`, make every **Checkpoint** pass, then run the final cell to write **`lab6_results.json`** |
| **2. Grader notebook** | Coursera Labs (no GPU) | For automated self assessment, paste the contents of `lab6_results.json`, then `Run All`. **40 ungraded points** |

The grader notebook has no GPU and no network, so it cannot re-run your
measurements. Instead it checks that your numbers are **internally consistent** —
each arithmetic intensity must follow from its own FLOP and byte counts, your
empirical ridge from the two peaks you measured, the Spyre ridge points from the
published specs, your landscape row from your raw Part 2 figures. Those relations
hold for any real run on any GPU, which is why a free-tier T4 earns a full score.

> **Students are encouraged to experiment with other GPUs and accelerators to expand their learning.** The free-tier T4 GPU is freely available on Google Colab. Re-running on an L4, V100 or A100 is genuinely instructive and you
> are encouraged to try it. There is **no extra credit** for doing so.

**Self-Assessment Lab ungraded points:**

| Part | Points | Comes from |
|---|---:|---|
| A — Structural invariants | 14 | Parts 1–6 of this notebook |
| B — Self-consistency | 18 | Parts 1–6 |
| C — Physics & plausibility | 8 | Parts 2–5 |
| **Ungraded notebook total** | **40** | fully auto-graded from your JSON for Self-Assessment |

**Graded Quiz points:**  
A 10 point graded Coursera Assignment quiz is given to assess your understanding from this lab experience.

| Quiz | Points | Comes from |
|---|---:|---|
| Concept Check quiz | 10 | Auto-graded Coursera **Graded Assignment** |

The **Checkpoint** cells in this notebook are self-checks: they catch mistakes
as you work and assign no grade. Treat them as a preview — if a Checkpoint
fails here, the corresponding grader cell will fail there too.

### Working through this lab
1. Set **Runtime → Change runtime type → T4 GPU** before you run anything.
2. Run each cell in order. Make sure every **Checkpoint** cell passes (no `AssertionError`).
3. Fill in every `TODO`.
4. Run the final cell to produce **`lab6_results.json`**, then open the grader
   notebook on Coursera and paste it in for self-assessment.
5. Take the **Concept Check quiz** — a separate auto-graded Coursera
   Assignment worth **10 graded points**. It is mandatory, and you need 80% to
   pass.


## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                           ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


---
## Setup
Run the cell below to install dependencies and detect your GPU.

In [ ]:
# Shared hardware reference. identical across every lab in this course.
# ridge = peak FP16 TFLOPS / peak memory bandwidth (TB/s): the arithmetic intensity
# (FLOPs/byte) where a workload flips from memory-bound to compute-bound. These match
# the lecture decks (T4 ridge ~203, H100 ~295).
HW_SPECS = {
    'Tesla T4':              {'fp16_tflops': 65.0,  'bw_tb_s': 0.320, 'ridge': 203},
    'NVIDIA A100-SXM4-40GB': {'fp16_tflops': 312.0, 'bw_tb_s': 1.555, 'ridge': 201},
    'NVIDIA A100-SXM4-80GB': {'fp16_tflops': 312.0, 'bw_tb_s': 2.039, 'ridge': 153},
    'NVIDIA H100':           {'fp16_tflops': 989.0, 'bw_tb_s': 3.35,  'ridge': 295},
}
# IBM Spyre (confirm against the current spec sheet before publishing):
SPYRE_SPECS = {'fp16_tflops': 98.0, 'lpddr5_tb_s': 0.204, 'scratchpad_mb': 64, 'scratchpad_tb_s': 10.0}

def lookup_hw(name):
    for _k, _v in HW_SPECS.items():
        if _k in name:
            return _k, _v
    return 'Tesla T4', HW_SPECS['Tesla T4']   # Colab free-tier default


> **Roofline & Arithmetic-Intensity Conventions (this notebook)**
> - **Precision:** FP16 (IEEE half) unless otherwise noted
> - **FLOPs counting:** 1 fused multiply-add = 2 FLOPs
> - **Memory level:** Device DRAM (HBM/GDDR), not L2 or scratchpad
> - **Byte convention:** Bytes *transferred* between DRAM and compute (reads + writes)
> - **Sparsity:** Dense (no structural sparsity) unless stated
> - **Hardware SKU:** Detected at runtime via `torch.cuda.get_device_name()`; specs from `hardware_profiles.json`

In [ ]:
!pip install -q transformers==4.44.2 matplotlib numpy

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.profiler import profile, ProfilerActivity, record_function
import time

# Detect GPU
assert torch.cuda.is_available(), "This lab requires a GPU runtime. Go to Runtime > Change runtime type > GPU."
device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
print(f"CUDA version: {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")

_hwname, _hw = lookup_hw(gpu_name)
PEAK_TFLOPS = _hw['fp16_tflops']; PEAK_BW_TB = _hw['bw_tb_s']; RIDGE_POINT = _hw['ridge']
if _hwname not in gpu_name:
    print(f"Note: unknown GPU '{gpu_name}'. Using {_hwname} specs as default.")

print(f"\nHardware Specs:")
print(f"  Peak FP16 Compute: {PEAK_TFLOPS} TFLOPS")
print(f"  Peak Memory BW:    {PEAK_BW_TB} TB/s")
print(f"  Ridge Point:       {RIDGE_POINT} FLOPs/byte")

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


---
## Part 1: Arithmetic Intensity from First Principles

Calculate the arithmetic intensity of three common DNN operators.

**Recall**: Arithmetic Intensity (AI) = FLOPs / Bytes transferred

For matrix multiply C = A × B:
- FLOPs = 2 × M × N × K (multiply-add for each output element)
- Bytes = (M×K + K×N + M×N) × bytes_per_element (read A, read B, write C)

In [ ]:
# TODO: Calculate arithmetic intensity for each operator
# All computations assume FP16 (2 bytes per element)

BYTES_PER_ELEM = 2  # FP16

# --- Operator 1: Large GEMM (Prefill FFN) ---
# Shape: [batch_size × seq_len, hidden] × [hidden, 4×hidden]
# M=2048 (batch*seq), K=4096 (hidden), N=16384 (4×hidden)
M1, K1, N1 = 2048, 4096, 16384

flops_gemm_large = None  # TODO: 2 * M1 * K1 * N1
bytes_gemm_large = None  # TODO: (M1*K1 + K1*N1 + M1*N1) * BYTES_PER_ELEM
ai_gemm_large = None     # TODO: flops / bytes

print(f"Operator 1 - Large GEMM (Prefill FFN):")
print(f"  Shape: [{M1},{K1}] × [{K1},{N1}]")
print(f"  FLOPs: {flops_gemm_large:,.0f}")
print(f"  Bytes: {bytes_gemm_large:,.0f}")
print(f"  AI:    {ai_gemm_large:.1f} FLOPs/byte")
print()

# --- Operator 2: Small GEMM (Decode, batch=1) ---
# Shape: [1, hidden] × [hidden, 4×hidden]; single-token decode
# M=1, K=4096, N=16384
M2, K2, N2 = 1, 4096, 16384

flops_gemm_small = None  # TODO
bytes_gemm_small = None  # TODO
ai_gemm_small = None     # TODO

print(f"Operator 2 - Small GEMM (Decode FFN, batch=1):")
print(f"  Shape: [{M2},{K2}] × [{K2},{N2}]")
print(f"  FLOPs: {flops_gemm_small:,.0f}")
print(f"  Bytes: {bytes_gemm_small:,.0f}")
print(f"  AI:    {ai_gemm_small:.2f} FLOPs/byte")
print()

# --- Operator 3: Attention QK^T (Decode) ---
# Shape: [1, head_dim] × [seq_len, head_dim]^T → [1, seq_len]
# This is a vector × matrix: FLOPs = 2 * 1 * seq_len * head_dim
seq_len, head_dim = 2048, 128

flops_attn = None   # TODO: 2 * 1 * seq_len * head_dim
bytes_attn = None   # TODO: (1*head_dim + seq_len*head_dim + 1*seq_len) * BYTES_PER_ELEM
ai_attn = None      # TODO

print(f"Operator 3 - Attention QK^T (Decode, single head):")
print(f"  Q shape: [1,{head_dim}], K shape: [{seq_len},{head_dim}]")
print(f"  FLOPs: {flops_attn:,.0f}")
print(f"  Bytes: {bytes_attn:,.0f}")
print(f"  AI:    {ai_attn:.2f} FLOPs/byte")

In [ ]:
# --- Checkpoint 1 ---
assert ai_gemm_large is not None, "Complete the large GEMM AI calculation"
assert ai_gemm_small is not None, "Complete the small GEMM AI calculation"
assert ai_attn is not None, "Complete the attention AI calculation"

# Verify ranges (not exact values; allow for rounding)
assert 100 < ai_gemm_large < 5000, f"Large GEMM AI={ai_gemm_large:.1f} seems wrong. Expected ~200-2000."
assert 0.5 < ai_gemm_small < 5, f"Small GEMM AI={ai_gemm_small:.2f} seems wrong. Expected ~1."
assert 0.1 < ai_attn < 3, f"Attention AI={ai_attn:.2f} seems wrong. Expected ~0.5-1."

print("[OK] Checkpoint 1 passed!")
print(f"\nKey insight: Prefill GEMM AI ({ai_gemm_large:.0f}) vs Decode GEMM AI ({ai_gemm_small:.2f})")
print(f"  → {ai_gemm_large/ai_gemm_small:.0f}× difference from batch size alone!")
print(f"  → Prefill is {'COMPUTE' if ai_gemm_large > RIDGE_POINT else 'MEMORY'}-bound on this GPU")
print(f"  → Decode is {'COMPUTE' if ai_gemm_small > RIDGE_POINT else 'MEMORY'}-bound on this GPU")

---
## Part 2: Empirical Roofline Construction

Measure the GPU's actual achievable peak compute and memory bandwidth, then plot the roofline.

We'll use:
- A large matrix multiply to measure peak compute
- A memory copy to measure peak bandwidth

### Measurement Best Practices

> **⚠️ Single-run timings can be misleading.** GPU performance varies due to thermal throttling,
> background processes, and memory state. Always use multiple measurements.

**Protocol for reliable benchmarks:**
1. **Warmup:** 3-5 iterations (discarded) to fill caches and trigger JIT compilation
2. **Repeat:** N ≥ 10 measurement iterations
3. **Report:** Median (robust to outliers) ± IQR or percentiles (P5, P50, P95)
4. **Synchronize:** Always call `torch.cuda.synchronize()` before timing GPU ops
5. **Isolate:** Use `torch.inference_mode()` or `torch.no_grad()` to avoid autograd overhead


In [ ]:
# TODO: Implement a robust benchmarking utility
# Requirements:
# 1. Accept a callable fn and run it n_warmup times (discard results)
# 2. Run fn n_measure times, recording wall-clock time for each
# 3. Return statistics: median, mean, std, P5, P95, IQR
# 4. Handle CUDA synchronization (torch.cuda.synchronize() before/after)
#
# Hint: Use time.perf_counter() for high-resolution timing
# Hint: np.percentile(times, [5, 50, 95]) gives you P5, median, P95

import time
import numpy as np

def bench_robust(fn, n_warmup=5, n_measure=20, sync_cuda=True):
    """Run fn() with warmup, return timing statistics in milliseconds."""
    # YOUR CODE HERE
    pass

# Test your implementation:
# x = torch.randn(1024, 1024, device=DEVICE)
# stats = bench_robust(lambda: torch.mm(x, x))
# assert 'median_ms' in stats
# assert stats['n'] == 20


In [ ]:
def measure_peak_compute(size=4096, dtype=torch.float16, warmup=5, trials=20):
    """Measure achieved compute via large GEMM (high AI → compute-bound)."""
    A = torch.randn(size, size, dtype=dtype, device=device)
    B = torch.randn(size, size, dtype=dtype, device=device)

    # Warmup
    for _ in range(warmup):
        torch.mm(A, B)
    torch.cuda.synchronize()

    # Measure
    times = []
    for _ in range(trials):
        torch.cuda.synchronize()
        start = time.perf_counter()
        torch.mm(A, B)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

    min_time = min(times)
    flops = 2 * size * size * size
    achieved_tflops = flops / min_time / 1e12
    return achieved_tflops


def measure_peak_bandwidth(size_mb=256, dtype=torch.float16, warmup=5, trials=20):
    """Measure achieved bandwidth via large memory copy."""
    num_elements = (size_mb * 1024 * 1024) // 2  # FP16 = 2 bytes
    src = torch.randn(num_elements, dtype=dtype, device=device)
    dst = torch.empty_like(src)

    # Warmup
    for _ in range(warmup):
        dst.copy_(src)
    torch.cuda.synchronize()

    # Measure
    times = []
    for _ in range(trials):
        torch.cuda.synchronize()
        start = time.perf_counter()
        dst.copy_(src)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

    min_time = min(times)
    total_bytes = 2 * num_elements * 2  # read + write
    achieved_bw_tb = total_bytes / min_time / 1e12
    return achieved_bw_tb


print("Measuring achieved peak compute (large GEMM)...")
achieved_compute = measure_peak_compute()
print(f"  Achieved: {achieved_compute:.1f} TFLOPS ({100*achieved_compute/PEAK_TFLOPS:.0f}% of spec)")

print("\nMeasuring achieved peak bandwidth (memory copy)...")
achieved_bw = measure_peak_bandwidth()
print(f"  Achieved: {achieved_bw:.3f} TB/s ({100*achieved_bw/PEAK_BW_TB:.0f}% of spec)")

achieved_ridge = achieved_compute / achieved_bw
print(f"\nEmpirical ridge point: {achieved_ridge:.0f} FLOPs/byte")
print(f"Spec ridge point:      {RIDGE_POINT} FLOPs/byte")

In [ ]:
# TODO: Plot the roofline model
# X-axis: Arithmetic Intensity (log scale, range 0.1 to 10000)
# Y-axis: Performance in TFLOPS (log scale)
# Plot: (1) bandwidth diagonal, (2) compute horizontal, (3) your 3 operators from Section 1

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Arithmetic intensity range (log scale)
ai_range = np.logspace(-1, 4, 1000)  # 0.1 to 10000

# TODO: Calculate roofline performance at each AI value
# roofline_perf = min(achieved_compute, achieved_bw * ai) for each ai value
roofline_perf = None  # TODO: np.minimum(achieved_compute, achieved_bw * ai_range * 1e12 / 1e12)
# Hint: achieved_bw is in TB/s, so BW * AI gives TFLOPS directly when AI is in FLOP/byte and BW is in TB/s

# Plot the roofline
ax.loglog(ai_range, roofline_perf, 'b-', linewidth=2, label='GPU Roofline')

# TODO: Plot the three operators from Section 1 as colored dots
# For each operator, its achieved performance is min(peak_compute, peak_bw * ai)
operators = {
    "Prefill FFN GEMM": ai_gemm_large,
    "Decode FFN GEMM": ai_gemm_small,
    "Decode Attention QK^T": ai_attn,
}

colors = ['green', 'red', 'orange']
for (name, ai), color in zip(operators.items(), colors):
    perf = min(achieved_compute, achieved_bw * ai)
    ax.plot(ai, perf, 'o', markersize=12, color=color, label=f"{name} (AI={ai:.1f})")

# Mark the ridge point
ax.axvline(x=achieved_ridge, color='gray', linestyle='--', alpha=0.5, label=f'Ridge = {achieved_ridge:.0f}')

ax.set_xlabel('Arithmetic Intensity (FLOPs/byte)', fontsize=12)
ax.set_ylabel('Performance (TFLOPS)', fontsize=12)
ax.set_title(f'Roofline Model; {gpu_name}', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.1, 10000)
ax.set_ylim(0.01, PEAK_TFLOPS * 2)

plt.tight_layout()
plt.savefig('roofline_gpu.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: roofline_gpu.png")

In [ ]:
# --- Checkpoint 2 ---
assert roofline_perf is not None, "Complete the roofline_perf calculation"
assert achieved_compute > 0.25 * PEAK_TFLOPS, f"Achieved compute ({achieved_compute:.1f}) seems too low. Check GPU runtime."
assert achieved_bw > 0.3 * PEAK_BW_TB, f"Achieved BW ({achieved_bw:.3f}) seems too low."

print("[OK] Checkpoint 2 passed!")
print(f"  Your GPU achieves {100*achieved_compute/PEAK_TFLOPS:.0f}% of spec compute and {100*achieved_bw/PEAK_BW_TB:.0f}% of spec bandwidth.")

---
## Part 3: LLM Kernel Profiling

Profile a real LLM running inference and extract per-kernel performance data.
We'll use GPT-2 (small enough for Colab) to demonstrate the pattern.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"  # 124M params; fits easily on T4
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
model.eval()

# Model specs
hidden = model.config.n_embd        # 768
n_layers = model.config.n_layer     # 12
n_heads = model.config.n_head       # 12
head_dim = hidden // n_heads        # 64
ffn_hidden = 4 * hidden             # 3072

print(f"Model: {model_name}")
print(f"  Hidden dim: {hidden}")
print(f"  Layers: {n_layers}")
print(f"  Heads: {n_heads}, Head dim: {head_dim}")
print(f"  FFN hidden: {ffn_hidden}")
print(f"  Total params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Profile prefill (all tokens processed at once)
prompt = "The roofline model is a visual performance analysis tool that helps engineers"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
seq_len_input = inputs["input_ids"].shape[1]
print(f"Input sequence length: {seq_len_input} tokens")

# Profile with FLOP counting
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    with_flops=True,
    profile_memory=True,
) as prof:
    with torch.no_grad():
        outputs = model(**inputs)

# Extract key events
events = prof.key_averages()
print(f"\nTop operators by CUDA time:")
sort_key = "cuda_time_total" if any(getattr(e, "cuda_time_total", 0) > 0 for e in events) else "cpu_time_total"
print(events.table(sort_by=sort_key, row_limit=15, top_level_events_only=False))

In [ ]:
# TODO: Extract arithmetic intensity data from profiler
# For each operator with FLOPs > 0, compute:
#   - FLOPs (from profiler)
#   - Time (CUDA time)
#   - Achieved TFLOPS = FLOPs / time
#   - Estimated AI (from shape analysis)

kernel_data = []  # list of dicts: {name, flops, time_us, achieved_tflops, estimated_ai}

for event in events:
    time_us = getattr(event, "cuda_time_total", 0) or event.self_cpu_time_total
    if event.flops and event.flops > 0 and time_us > 0:
        flops = event.flops
        time_s = time_us / 1e6
        achieved_tflops = flops / time_s / 1e12
        
        # TODO: Estimate bytes transferred for this op
        # Approximation: for GEMM [M,K]x[K,N], bytes ≈ (M*K + K*N + M*N) * 2
        # For ops where we can't determine shape, use: bytes ≈ flops / ridge_point
        # (This gives the "expected" bytes if the op were at the ridge)
        
        # Simple heuristic: use achieved performance to infer actual AI
        # If achieved << peak_compute, then op is memory-bound
        # estimated_ai ≈ achieved_tflops / achieved_bw
        estimated_ai = achieved_tflops / achieved_bw if achieved_bw > 0 else 0
        
        kernel_data.append({
            "name": event.key[:40],
            "flops": flops,
            "time_us": time_us,
            "achieved_tflops": achieved_tflops,
            "estimated_ai": estimated_ai,
        })

# Sort by time (most expensive first)
kernel_data.sort(key=lambda x: x["time_us"], reverse=True)

print(f"\nExtracted {len(kernel_data)} kernels with FLOP data:")
print(f"{'Kernel':<42} {'FLOPs':>12} {'Time(μs)':>10} {'TFLOPS':>8} {'Est.AI':>8}")
print("-" * 85)
for k in kernel_data[:10]:
    print(f"{k['name']:<42} {k['flops']:>12,.0f} {k['time_us']:>10.0f} {k['achieved_tflops']:>8.2f} {k['estimated_ai']:>8.1f}")

In [ ]:
# --- Checkpoint 3 ---
assert len(kernel_data) > 0, "No kernel data extracted. Make sure profiling ran correctly."
assert any(k["achieved_tflops"] > 0 for k in kernel_data), "No kernels achieved measurable TFLOPS."

print("[OK] Checkpoint 3 passed!")
print(f"  Profiled {len(kernel_data)} compute kernels.")
max_kernel = max(kernel_data, key=lambda x: x["achieved_tflops"])
print(f"  Highest achiever: {max_kernel['name']} at {max_kernel['achieved_tflops']:.2f} TFLOPS")

---
## Part 4: Roofline Diagnosis

Plot profiled kernels on the roofline and classify each as compute-bound or memory-bound.
Then propose one optimization for the most time-consuming memory-bound kernel.

In [ ]:
# TODO: Plot profiled kernels on the roofline from Section 2

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot roofline
ai_range = np.logspace(-1, 4, 1000)
roofline = np.minimum(achieved_compute, achieved_bw * ai_range)
ax.loglog(ai_range, roofline, 'b-', linewidth=2, label='Roofline', alpha=0.7)

# TODO: Plot each profiled kernel as a dot
# X = estimated_ai, Y = achieved_tflops
# Color by classification: green=compute-bound (near ceiling), red=memory-bound (below ceiling)

for k in kernel_data[:15]:  # Top 15 by time
    ai = k["estimated_ai"]
    perf = k["achieved_tflops"]
    if ai > 0 and perf > 0:
        # Classify: if achieved > 50% of roofline at this AI, it's well-optimized
        roof_at_ai = min(achieved_compute, achieved_bw * ai)
        efficiency = perf / roof_at_ai if roof_at_ai > 0 else 0
        color = 'green' if efficiency > 0.5 else 'red'
        ax.plot(ai, perf, 'o', color=color, markersize=8, alpha=0.7)
        ax.annotate(k["name"][:15], (ai, perf), fontsize=7, alpha=0.8)

ax.axvline(x=achieved_ridge, color='gray', linestyle='--', alpha=0.5, label=f'Ridge={achieved_ridge:.0f}')
ax.set_xlabel('Arithmetic Intensity (FLOPs/byte)', fontsize=12)
ax.set_ylabel('Achieved Performance (TFLOPS)', fontsize=12)
ax.set_title(f'LLM Kernel Roofline; {model_name} on {gpu_name}', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0.1, 10000)

plt.tight_layout()
plt.savefig('roofline_kernels.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# TODO: Fill in the classification table
# For each of the top 5 most time-consuming kernels:
# 1. Is it compute-bound or memory-bound?  (compare its AI to achieved_ridge)
# 2. What fraction of the roofline ceiling does it achieve?
#
# Note on the AI estimate: Part 3 derived estimated_ai as
# achieved_tflops / achieved_bw, which PINS every point to the bandwidth
# diagonal by construction. That is why the Efficiency column below reads 100%
# for every row. It is a teaching heuristic, not a real diagnostic -- a true AI
# needs the per-op byte count from shape analysis.

print("Kernel Classification Table:")
print(f"{'Kernel':<35} {'AI':>6} {'Achieved':>9} {'Ceiling':>9} {'Efficiency':>10} {'Bound':>10}")
print("-" * 85)

for k in kernel_data[:5]:
    ai = k["estimated_ai"]
    perf = k["achieved_tflops"]
    ceiling = min(achieved_compute, achieved_bw * ai) if ai > 0 else achieved_compute
    efficiency = perf / ceiling if ceiling > 0 else 0
    bound = "COMPUTE" if ai > achieved_ridge else "MEMORY"
    print(f"{k['name'][:35]:<35} {ai:>6.1f} {perf:>8.2f}T {ceiling:>8.2f}T {efficiency:>9.0%} {bound:>10}")


In [ ]:
# 4.2: * TODO - record your optimization choice as a structured answer.
#
# Part 4's table named the most time-consuming MEMORY-bound kernel. Now commit
# to ONE optimization for it, and to what that optimization does on the
# roofline. Both answers come from a controlled vocabulary, so they are
# auto-checkable -- no prose, no human review.
#
# OPTIMIZATION_CHOICE  -- exactly one of:
#     "quantization"       weights FP16 -> INT8/INT4: fewer bytes per MAC
#     "operator_fusion"    keep intermediates on-chip between elementwise ops
#     "increase_batch"     reuse each loaded weight across more tokens
#     "tiling_for_cache"   block the operand so a tile is reread from L2/SRAM
#
# OPTIMIZATION_EFFECT  -- exactly one of, describing what it does on the plot:
#     "right_and_up_along_bandwidth_diagonal"   raises AI; slides the point
#                                               right along the BW roof
#     "raises_the_compute_roof"                 lifts the horizontal ceiling
#     "raises_the_bandwidth_roof"               lifts the diagonal
#     "no_change"                               moves nothing
#
# Think it through: a weight-bound GEMM at AI ~= 1 is limited by the bytes it
# reads, not by the FLOPs it issues. Which choice changes the BYTES, and does
# any choice change the ROOF itself?

OPTIMIZATION_CHOICE = None   # TODO: one of the four strings above
OPTIMIZATION_EFFECT = None   # TODO: one of the four strings above

print(f"Optimization choice: {OPTIMIZATION_CHOICE}")
print(f"Effect on roofline:  {OPTIMIZATION_EFFECT}")


In [ ]:
# --- Checkpoint 4 ---
assert len(kernel_data) >= 1, "No kernels classified. Re-run Part 3."
assert OPTIMIZATION_CHOICE in {
    "quantization", "operator_fusion", "increase_batch", "tiling_for_cache"
}, f"OPTIMIZATION_CHOICE must be one of the four offered strings, got {OPTIMIZATION_CHOICE!r}."
assert OPTIMIZATION_EFFECT in {
    "right_and_up_along_bandwidth_diagonal", "raises_the_compute_roof",
    "raises_the_bandwidth_roof", "no_change",
}, f"OPTIMIZATION_EFFECT must be one of the four offered strings, got {OPTIMIZATION_EFFECT!r}."

print("[OK] Checkpoint 4 passed!")
print(f"  Classified {min(len(kernel_data), 5)} kernels; "
      f"optimization = {OPTIMIZATION_CHOICE}, effect = {OPTIMIZATION_EFFECT}")


---
## Part 5: Dual Roofline: GPU vs. Spyre Crossover

Construct a dual roofline comparing your GPU with IBM Spyre's scratchpad-based architecture.
Identify which operators from Section 3 would benefit from running on Spyre.

**Spyre specs**:
- Peak FP16 compute: 98 TFLOPS
- External memory (LPDDR5): 204 GB/s
- On-chip scratchpad: 64 MB, effective bandwidth >> 204 GB/s
- For this exercise, assume scratchpad effective bandwidth of **10 TB/s** (conservative estimate for ring interconnect across 32 cores)

In [ ]:
# Spyre specs
SPYRE_PEAK_TFLOPS = 98.0
SPYRE_LPDDR5_BW = 0.204  # TB/s
SPYRE_SCRATCHPAD_BW = 10.0  # TB/s (conservative estimate for on-chip)
SPYRE_SCRATCHPAD_SIZE_MB = 64

# Ridge points
spyre_ridge_lpddr5 = SPYRE_PEAK_TFLOPS / SPYRE_LPDDR5_BW
spyre_ridge_scratchpad = SPYRE_PEAK_TFLOPS / SPYRE_SCRATCHPAD_BW

print(f"Spyre Ridge Points:")
print(f"  LPDDR5 level:     {spyre_ridge_lpddr5:.0f} FLOPs/byte")
print(f"  Scratchpad level: {spyre_ridge_scratchpad:.0f} FLOPs/byte")
print(f"  GPU ridge:        {achieved_ridge:.0f} FLOPs/byte")

# Dual roofline:
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

ai_range = np.logspace(-1, 4, 1000)

# GPU roofline (your measured GPU)
gpu_roof = np.minimum(achieved_compute, achieved_bw * ai_range)
ax.loglog(ai_range, gpu_roof, 'b-', linewidth=2.5, label=f'GPU ({gpu_name})')

# Spyre roofline, scratchpad level (data that fits in 64 MB):
spyre_scratchpad_roof = np.minimum(SPYRE_PEAK_TFLOPS, SPYRE_SCRATCHPAD_BW * ai_range)

# Spyre roofline, LPDDR5 level (data that does not fit):
spyre_lpddr5_roof = np.minimum(SPYRE_PEAK_TFLOPS, SPYRE_LPDDR5_BW * ai_range)

ax.loglog(ai_range, spyre_scratchpad_roof, 'g-', linewidth=2.5,
          label=f'Spyre (scratchpad, {SPYRE_SCRATCHPAD_SIZE_MB}MB on-chip)')
ax.loglog(ai_range, spyre_lpddr5_roof, 'g--', linewidth=1.5, alpha=0.6,
          label='Spyre (LPDDR5 fallback)')

# Find crossover point (where Spyre scratchpad > GPU)
# The AI value where spyre_scratchpad_roof > gpu_roof:
crossover_mask = spyre_scratchpad_roof > gpu_roof
if crossover_mask.any():
    crossover_ai = ai_range[crossover_mask][-1]  # highest AI where Spyre wins
    ax.axvline(x=crossover_ai, color='purple', linestyle=':', alpha=0.7,
              label=f'Crossover AI ≈ {crossover_ai:.1f}')
    ax.fill_between(ai_range, 0.001, 1000, where=crossover_mask,
                   alpha=0.05, color='green')

# Plot decode operators from Section 1
ax.plot(ai_gemm_small, min(achieved_compute, achieved_bw * ai_gemm_small),
       's', markersize=14, color='red', label=f'Decode FFN (AI={ai_gemm_small:.1f})', zorder=5)
ax.plot(ai_attn, min(achieved_compute, achieved_bw * ai_attn),
       '^', markersize=14, color='orange', label=f'Decode Attn (AI={ai_attn:.2f})', zorder=5)

ax.set_xlabel('Arithmetic Intensity (FLOPs/byte)', fontsize=12)
ax.set_ylabel('Performance (TFLOPS)', fontsize=12)
ax.set_title('Dual Roofline: GPU vs. IBM Spyre', fontsize=14)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.1, 10000)
ax.set_ylim(0.01, max(PEAK_TFLOPS, SPYRE_PEAK_TFLOPS) * 2)

plt.tight_layout()
plt.savefig('roofline_dual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5.2: * TODO - the capacity arithmetic behind the Spyre crossover.
#
# Part 5's plot showed the Spyre scratchpad roofline sitting above the GPU
# roofline across the whole AI sweep. That is the trap in this lab: there is no
# interior AI crossover. The real threshold is CAPACITY -- does the working set
# fit in the 64 MB scratchpad? Work that out numerically.
#
# All four values below are exact arithmetic. None of them depend on which GPU
# Colab gave you, except DECODE_ADVANTAGE_X, which uses YOUR measured achieved_bw.

# (a) How many FP16 parameters fit in the scratchpad at once?
#     Hint: SPYRE_SCRATCHPAD_SIZE_MB MB, at 2 bytes per parameter. Use
#     1 MB = 1024*1024 bytes, consistent with the 64 MB figure as a hardware size.
MAX_PARAMS_ON_CHIP = None   # TODO: int

# (b) A 7B-parameter model in FP16 is how many bytes?
MODEL_7B_FP16_BYTES = None  # TODO: int, 7e9 params x 2 bytes

# (c) Do those weights fit on-chip?
MODEL_7B_FITS = None        # TODO: bool

# (d) At decode AI (ai_gemm_small ~= 1), how much faster is Spyre's scratchpad
#     than the GPU memory you measured? Both are memory-bound there, so
#     attainable performance is bandwidth x AI and the AI cancels: the ratio is
#     just the bandwidth ratio.
#     Hint: SPYRE_SCRATCHPAD_BW / achieved_bw   (both already in TB/s)
DECODE_ADVANTAGE_X = None   # TODO: float

# (e) Which kind of threshold actually gates the Spyre win? One of:
#       "arithmetic_intensity"   a crossover at some AI value
#       "capacity"               whether the working set fits in 64 MB
#       "precision"              the numeric format
#       "clock_frequency"        core clock
CROSSOVER_KIND = None       # TODO: one of the four strings above

print(f"Params on-chip at once : {MAX_PARAMS_ON_CHIP:,}" if MAX_PARAMS_ON_CHIP else "TODO")
print(f"7B model in FP16       : {MODEL_7B_FP16_BYTES:,} bytes" if MODEL_7B_FP16_BYTES else "TODO")
print(f"Fits in scratchpad?    : {MODEL_7B_FITS}")
print(f"Decode BW advantage    : {DECODE_ADVANTAGE_X:.1f}x" if DECODE_ADVANTAGE_X else "TODO")
print(f"Crossover is gated by  : {CROSSOVER_KIND}")


In [ ]:
# --- Checkpoint 5 ---
assert spyre_scratchpad_roof is not None, "Complete the Spyre scratchpad roofline calculation."
assert spyre_lpddr5_roof is not None, "Complete the Spyre LPDDR5 roofline calculation."
assert MAX_PARAMS_ON_CHIP == int(SPYRE_SCRATCHPAD_SIZE_MB * 1024 * 1024 / 2), (
    f"MAX_PARAMS_ON_CHIP should be {int(SPYRE_SCRATCHPAD_SIZE_MB*1024*1024/2):,} "
    f"(64 MB / 2 bytes per FP16 param), got {MAX_PARAMS_ON_CHIP}.")
assert MODEL_7B_FITS is False, (
    "A 7B FP16 model is 14 GB and the scratchpad is 64 MB, so MODEL_7B_FITS is False.")
assert DECODE_ADVANTAGE_X > 1.0, (
    f"Spyre's 10 TB/s scratchpad beats any GPU's DRAM at decode AI, so the "
    f"advantage is > 1x, got {DECODE_ADVANTAGE_X}.")
assert CROSSOVER_KIND == "capacity", (
    "The Spyre win is gated by whether the working set fits in 64 MB, not by "
    "an arithmetic-intensity crossover.")

print("[OK] Checkpoint 5 passed!")
print(f"  Scratchpad holds {MAX_PARAMS_ON_CHIP:,} FP16 params; "
      f"decode advantage {DECODE_ADVANTAGE_X:.1f}x; gated by {CROSSOVER_KIND}.")


### Prefill vs. Decode: Separate Roofline Analysis

LLM inference has two distinct phases with very different computational profiles:

| Phase | Characteristic | Arithmetic Intensity | Roofline Position |
|---|---|---|---|
| **Prefill** | Process full prompt in parallel | High (compute-bound) | Near/above ridge point |
| **Decode** | Generate one token at a time | Low (memory-bound) | Left of ridge point |

**Why separate them?**
- Prefill: large batch of tokens → high reuse of weights → compute-bound
- Decode: single token per step → each weight loaded once per token → memory-bound
- A single "inference AI" number hides this critical distinction

In [ ]:
# TODO: Compute separate arithmetic intensity for prefill and decode phases
#
# Prefill: process seq_len tokens in parallel
#   - FLOPs: count all matmuls in attention (Q,K,V,O projections + scores) and FFN
#   - Bytes: weight parameters loaded once (amortized over seq_len tokens)
#
# Decode: generate 1 token, attending to the full KV cache
#   - FLOPs: same matmuls but for a single token (seq_len=1 for projections)
#   - Bytes: weights loaded once + the KV cache read for attention
#
# Key insight: prefill reuses weights across many tokens (high AI);
# decode loads the weights for just one token (low AI).
#
# Both AI values are PURE ARITHMETIC from the config below, so they are the
# same on every GPU -- which is why the grader can check them exactly.
#
# Part 7 exports these names, so define all of them:
#   params_per_layer, prefill_flops_total, prefill_bytes, prefill_ai,
#   decode_flops_total, decode_bytes, kv_cache_bytes_read, decode_ai

d_model = 768
n_heads = 12
n_layers = 12
seq_len = 512      # prompt length
dtype_bytes = 2    # FP16

# --- PREFILL phase ---
# Per layer, over the full sequence (x2 everywhere because 1 FMA = 2 FLOPs):
#   attention projections : 4 * seq_len * d_model * d_model * 2
#   attention scores      : 2 * seq_len * seq_len * d_model * 2
#   FFN (two linears)     : 2 * seq_len * d_model * (4*d_model) * 2
attn_proj_flops  = None   # TODO
attn_score_flops = None   # TODO
ffn_flops        = None   # TODO

prefill_flops_per_layer = None   # TODO: sum of the three above
prefill_flops_total     = None   # TODO: x n_layers

# Bytes: the weights are loaded once and reused across all seq_len tokens.
params_per_layer = None   # TODO: 4 * d_model**2 + 2 * d_model * (4*d_model)
prefill_bytes    = None   # TODO: params_per_layer * n_layers * dtype_bytes

prefill_ai = None         # TODO: prefill_flops_total / prefill_bytes

# --- DECODE phase (a single token) ---
decode_seq_len = 1
attn_proj_flops_dec  = None   # TODO: 4 * decode_seq_len * d_model * d_model * 2
attn_score_flops_dec = None   # TODO: 2 * decode_seq_len * seq_len * d_model * 2
ffn_flops_dec        = None   # TODO: 2 * decode_seq_len * d_model * (4*d_model) * 2

decode_flops_per_layer = None   # TODO
decode_flops_total     = None   # TODO: x n_layers

# Bytes: the SAME weights, but amortized over one token only, plus the KV
# cache read:  2 (K and V) * n_layers * seq_len * d_model * dtype_bytes
kv_cache_bytes_read = None   # TODO
decode_bytes        = None   # TODO: weights + kv_cache_bytes_read

decode_ai = None             # TODO: decode_flops_total / decode_bytes

print(f"{'Phase':<10} {'FLOPs':>14} {'Bytes':>14} {'AI (FLOPs/B)':>14} {'Regime'}")
print(f"{'-'*10} {'-'*14} {'-'*14} {'-'*14} {'-'*15}")
print(f"{'Prefill':<10} {prefill_flops_total:>14,.0f} {prefill_bytes:>14,.0f} "
      f"{prefill_ai:>14.1f} {'compute-bound' if prefill_ai > achieved_ridge else 'memory-bound'}")
print(f"{'Decode':<10} {decode_flops_total:>14,.0f} {decode_bytes:>14,.0f} "
      f"{decode_ai:>14.1f} {'compute-bound' if decode_ai > achieved_ridge else 'memory-bound'}")

print(f"\nEmpirical ridge point ({gpu_name}): {achieved_ridge:.0f} FLOPs/byte")
print(f"  Prefill AI ({prefill_ai:.1f}) {'>' if prefill_ai > achieved_ridge else '<'} ridge "
      f"-> {'COMPUTE-bound' if prefill_ai > achieved_ridge else 'MEMORY-bound'}")
print(f"  Decode AI  ({decode_ai:.1f}) {'>' if decode_ai > achieved_ridge else '<'} ridge "
      f"-> {'COMPUTE-bound' if decode_ai > achieved_ridge else 'MEMORY-bound'}")


In [ ]:
# --- Checkpoint 5b ---
assert prefill_ai > 100, f"Prefill AI should be high (compute-bound), got {prefill_ai:.1f}"
assert decode_ai < 100, f"Decode AI should be low (memory-bound), got {decode_ai:.1f}"
assert abs(decode_ai - 1.0) < 0.3, (
    f"Decode AI should land near 1.0 FLOP/byte -- one pass over the weights per "
    f"token -- got {decode_ai:.2f}.")
assert prefill_ai / decode_ai > 100, (
    "Prefill should out-reuse decode by orders of magnitude; check that the "
    "prefill byte count amortises the weights over all seq_len tokens.")

print("[OK] Checkpoint 5b passed!")
print(f"  Prefill AI {prefill_ai:.1f} (compute-bound) vs decode AI "
      f"{decode_ai:.2f} (memory-bound): {prefill_ai/decode_ai:.0f}x apart.")


---

## Part 6 - Update the Accelerator Landscape Map

Add this week's row to the shared map you have been building since Lab 1.


In [ ]:
# Part 6 -- Landscape Map row for Week 6. Fill the measured values from your run.
landscape_row_w6 = {
    "accelerator":          gpu_name,
    "achieved_TFLOPS":      None,   # TODO: round(achieved_compute, 1)
    "achieved_BW_TB_s":     None,   # TODO: round(achieved_bw, 3)
    "empirical_ridge":      None,   # TODO: round(achieved_ridge, 0)
    "decode_FFN_AI":        None,   # TODO: round(ai_gemm_small, 2)
    "decode_bound":         None,   # TODO: "memory" if ai_gemm_small < achieved_ridge else "compute"
    "spyre_scratchpad_win": "when the working set fits in 64 MB (capacity-gated)",
}
for _k, _v in landscape_row_w6.items():
    print(f"  {_k:22s}: {_v}")


In [ ]:
# --- Checkpoint 6 ---
_req = {"accelerator", "achieved_TFLOPS", "achieved_BW_TB_s", "empirical_ridge",
        "decode_FFN_AI", "decode_bound", "spyre_scratchpad_win"}
assert set(landscape_row_w6) == _req, (
    f"landscape_row_w6 should have exactly these keys: {sorted(_req)}")
assert all(v is not None and v != "" for v in landscape_row_w6.values()), (
    "Every field in landscape_row_w6 must be filled in.")
assert abs(landscape_row_w6["achieved_TFLOPS"] - round(achieved_compute, 1)) < 0.05, \
    "achieved_TFLOPS must mirror your measured achieved_compute."
assert abs(landscape_row_w6["achieved_BW_TB_s"] - round(achieved_bw, 3)) < 5e-4, \
    "achieved_BW_TB_s must mirror your measured achieved_bw."
assert landscape_row_w6["decode_bound"] == "memory", (
    "At AI ~= 1 the decode FFN GEMM sits far left of any GPU's ridge point.")

print("[OK] Checkpoint 6 passed!")
print("  Landscape Map row complete and consistent with your Part 2 measurements.")


---

## Part 7: Save and export your results

The cell below writes **`lab6_results.json`** and prints it. Students copy its
contents into the `RESULTS_JSON` cell of the **grader notebook** on Coursera
Labs and run every cell there to see their **40 ungraded Self-Assessment
points**.

It carries the **raw inputs** next to every derived number — the FLOP and byte
counts behind each arithmetic intensity, the two measured peaks behind the
empirical ridge, the Spyre specs behind both Spyre ridge points — so the grader
can recompute each derived value rather than take it on trust.

> Those 40 points are **feedback, not a grade** — they tell you how well the lab
> went. The graded item for this week is the **Concept Check quiz** (10 points,
> auto-graded on Coursera, 80% to pass).


In [ ]:
# === Part 7: Save your results for submission ==========================
# This writes lab6_results.json. You will paste its contents into the
# SELF-ASSESSMENT notebook on Coursera Labs ("Lab 6 - Self-Assessment Check"),
# which reports the 40 ungraded Self-Assessment points. Without it there is
# nothing to check.
#
# PROVIDED - do not edit. Everything here is built from variables you already
# computed in Parts 1-6, so there is nothing to fill in and nothing to get
# wrong. Just run it.
#
# It carries the RAW INPUTS next to every derived number -- the FLOP and byte
# counts, the measured peaks, your model config -- so the self-assessment
# notebook can RECOMPUTE each derived value rather than take it on trust.
import json as _json

# Part 1 reassigns `seq_len` and `head_dim` later in the notebook (Part 3 uses
# 512 / 64), so pin the Part 1 operand shapes explicitly here rather than
# reading whatever those names hold now.
_ATTN_SEQ_LEN, _ATTN_HEAD_DIM = 2048, 128

RESULTS = {
    # --- Part 1: arithmetic intensity from first principles --------------
    # Raw FLOP and byte counts travel next to each AI so the check can divide
    # them itself. These are pure arithmetic: identical on every GPU.
    "part1_operators": {
        "gemm_large": {                       # prefill FFN GEMM
            "shape": [M1, K1, N1],
            "flops": int(flops_gemm_large),
            "bytes": int(bytes_gemm_large),
            "ai":    float(ai_gemm_large),
        },
        "gemm_small": {                       # decode FFN GEMM, batch=1
            "shape": [M2, K2, N2],
            "flops": int(flops_gemm_small),
            "bytes": int(bytes_gemm_small),
            "ai":    float(ai_gemm_small),
        },
        "attn_qk": {                          # decode attention QK^T, 1 head
            "seq_len":  int(_ATTN_SEQ_LEN),
            "head_dim": int(_ATTN_HEAD_DIM),
            "flops":    int(flops_attn),
            "bytes":    int(bytes_attn),
            "ai":       float(ai_attn),
        },
        "bytes_per_elem": int(BYTES_PER_ELEM),
    },

    # --- Part 2: the empirical roofline you measured ---------------------
    # These ARE hardware-dependent. The checks band them and verify the ridge
    # follows from the two peaks, never that they equal a T4's numbers.
    "part2_roofline": {
        "gpu_name":          str(gpu_name),
        "spec_tflops":       float(PEAK_TFLOPS),
        "spec_bw_tb_s":      float(PEAK_BW_TB),
        "spec_ridge":        float(RIDGE_POINT),
        "achieved_tflops":   float(achieved_compute),
        "achieved_bw_tb_s":  float(achieved_bw),
        "achieved_ridge":    float(achieved_ridge),
    },

    # --- Part 3: the profiled LLM kernels -------------------------------
    # Which kernels appear depends on the GPU and the transformers version, so
    # the checks look at the SHAPE of this data and its internal arithmetic.
    "part3_profile": {
        "model_name":     str(model_name),
        "hidden":         int(hidden),
        "n_layers":       int(n_layers),
        "n_heads":        int(n_heads),
        "head_dim":       int(hidden // n_heads),
        "ffn_hidden":     int(ffn_hidden),
        "n_params":       int(sum(p.numel() for p in model.parameters())),
        "seq_len_input":  int(seq_len_input),
        "n_kernels":      int(len(kernel_data)),
        # Top kernels by time, with the raw FLOPs and microseconds that the
        # achieved TFLOPS was computed from.
        "kernels": [
            {
                "name":            str(k["name"]),
                "flops":           int(k["flops"]),
                "time_us":         float(k["time_us"]),
                "achieved_tflops": float(k["achieved_tflops"]),
                "estimated_ai":    float(k["estimated_ai"]),
            }
            for k in kernel_data[:10]
        ],
    },

    # --- Part 4: roofline diagnosis -------------------------------------
    # The classification of each kernel, derived from YOUR measured ridge.
    "part4_diagnosis": {
        "classification": [
            {
                "name":       str(k["name"]),
                "ai":         float(k["estimated_ai"]),
                "achieved":   float(k["achieved_tflops"]),
                "ceiling":    float(min(achieved_compute, achieved_bw * k["estimated_ai"])
                                    if k["estimated_ai"] > 0 else achieved_compute),
                "bound":      ("COMPUTE" if k["estimated_ai"] > achieved_ridge else "MEMORY"),
            }
            for k in kernel_data[:5]
        ],
        # Controlled vocabulary: which optimization you chose for the most
        # expensive memory-bound kernel. One of the four offered in Part 4.
        "optimization_choice": OPTIMIZATION_CHOICE,
        "optimization_effect": OPTIMIZATION_EFFECT,
    },

    # --- Part 5: the GPU vs Spyre dual roofline -------------------------
    # The two Spyre ridge points are exact arithmetic from the published specs,
    # so they are checked by equality. The capacity answers are too.
    "part5_spyre": {
        "spyre_peak_tflops":      float(SPYRE_PEAK_TFLOPS),
        "spyre_lpddr5_bw":        float(SPYRE_LPDDR5_BW),
        "spyre_scratchpad_bw":    float(SPYRE_SCRATCHPAD_BW),
        "spyre_scratchpad_mb":    float(SPYRE_SCRATCHPAD_SIZE_MB),
        "ridge_lpddr5":           float(spyre_ridge_lpddr5),
        "ridge_scratchpad":       float(spyre_ridge_scratchpad),
        # Capacity arithmetic: 64 MB / 2 bytes = 32 M parameters on-chip.
        "max_params_on_chip":     int(MAX_PARAMS_ON_CHIP),
        # Does a 7B FP16 model fit on-chip? (It does not -- by ~220x.)
        "model_7b_fp16_bytes":    int(MODEL_7B_FP16_BYTES),
        "model_7b_fits_on_chip":  bool(MODEL_7B_FITS),
        # Spyre's scratchpad advantage at decode AI, vs YOUR measured GPU BW.
        "decode_advantage_x":     float(DECODE_ADVANTAGE_X),
        # Which memory level gates the win: capacity, not an AI crossover.
        "crossover_kind":         CROSSOVER_KIND,
    },

    # --- Part 5b: prefill vs decode, separated --------------------------
    # Pure arithmetic from the GPT-2 config: identical on every GPU.
    "part5b_phases": {
        "d_model":        int(d_model),
        "n_layers":       int(n_layers),
        "seq_len":        int(seq_len),
        "dtype_bytes":    int(dtype_bytes),
        "params_per_layer": int(params_per_layer),
        "prefill": {
            "flops": int(prefill_flops_total),
            "bytes": int(prefill_bytes),
            "ai":    float(prefill_ai),
            "bound": ("COMPUTE" if prefill_ai > achieved_ridge else "MEMORY"),
        },
        "decode": {
            "flops":         int(decode_flops_total),
            "bytes":         int(decode_bytes),
            "kv_bytes_read": int(kv_cache_bytes_read),
            "ai":            float(decode_ai),
            "bound":         ("COMPUTE" if decode_ai > achieved_ridge else "MEMORY"),
        },
    },

    # --- Part 6: your Landscape Map row ---------------------------------
    # Rounded display values on purpose; the checks confirm they round-trip
    # against the raw Part 2 measurements.
    "landscape_row_w6": landscape_row_w6,

    # --- environment: reported, never graded against --------------------
    "environment": {
        "device":        (torch.cuda.get_device_name(0)
                          if torch.cuda.is_available() else "CPU only"),
        "torch_version": torch.__version__,
        "mode":          MODE,
    },
}

with open("lab6_results.json", "w") as _f:
    _json.dump(RESULTS, _f, indent=2)

print("Saved lab6_results.json\n")
print(_json.dumps(RESULTS, indent=2))
print("\n" + "=" * 70)
print("NEXT STEPS")
print("=" * 70)
print("1. Copy the JSON printed above (from the opening '{' through the")
print("   final '}'), or open the file browser (folder icon, left sidebar)")
print("   and double-click lab6_results.json.")
print("2. Open the SELF-ASSESSMENT notebook on Coursera Labs.")
print("3. Paste the contents into the RESULTS_JSON cell, then Run All.")
print("   It reports your 40 ungraded Self-Assessment points as feedback.")
print("4. The Concept Check quiz is the GRADED item: a separate auto-graded")
print("   Coursera Assignment worth 10 points (80% to pass). Take it as well.")
print("\nKeep this Colab notebook: if a self-assessment check fails you will")
print("want to re-run the cell that produced the value.")

# Optional: download the file instead of copy-pasting.
# from google.colab import files
# files.download('lab6_results.json')


---

## Check your understanding (ungraded)

These prompts are **not graded and not submitted** — nothing here travels in your
JSON, and no one reviews your answers. Work through them anyway: they are the
reasoning the **Concept Check quiz** tests, and that quiz *is* the graded one,
worth **10 graded points**. Answer in your own words, citing the numbers *you*
measured, then compare against the solution notebook once it unlocks.

---

### 1. Why is your empirical ridge point roughly half the spec ridge point?

You measured `achieved_tflops` and `achieved_bw_tb_s`, then divided them to get
`achieved_ridge`. Work out each as a *percentage of spec* from the figures Part 2
printed. On a reference free-tier T4 run they came out very unevenly — about 39%
of the compute spec but 74% of the bandwidth spec — which drags the ridge down to
roughly half the datasheet value.

Explain why a large GEMM is so much harder to drive to its peak than a `copy_`
is. Then answer the general version: the ridge is a *ratio*, so which way does it
move if a GPU reaches a *higher* fraction of peak compute than of peak bandwidth?

*Your reasoning here (ungraded — for your own learning).*

---

### 2. The same GEMM, two verdicts.

Part 1's large GEMM has AI ≈ 1260 and the decode GEMM has AI ≈ 1.0 — a factor of
~1260 apart — yet they multiply by the *same* weight matrix. Only `M` changed,
from 2048 to 1. Explain where the factor of ~1260 comes from in the
FLOPs/bytes ratio, and why batching is therefore the single biggest lever on
decode throughput. Then say what that implies for a serving system that must
answer one user at a time.

*Your reasoning here (ungraded — for your own learning).*

---

### 3. Why the Part 4 efficiency column reads 100% for every kernel.

Part 3 estimated each kernel's AI as `achieved_tflops / achieved_bw`. Substitute
that definition into the ceiling formula `min(peak, bw × AI)` and show
algebraically why the efficiency must come out at 100% whenever the kernel is
below the ridge. Then name what you would actually need to measure to place a
kernel on the roofline honestly.

*Your reasoning here (ungraded — for your own learning).*

---

### 4. The capacity question (the Landscape Map question).

You found that a 7B FP16 model needs 14 GB against Spyre's 64 MB scratchpad, and
that 64 MB holds 33,554,432 parameters at once. One GPT-2 transformer layer is
7,077,888 parameters — 13.5 MB in FP16 — so a single layer fits comfortably, while
the whole 124 M-parameter model at 237 MB does not.

Name the decision this forces on the Spyre compiler that a GPU never has to
make, and say what it costs when it gets it wrong. Then explain why Spyre's
LPDDR5 bandwidth (0.204 TB/s) being *below* your measured GPU bandwidth means a
model that misses the scratchpad is slower on Spyre, not faster — and what that
says about where dataflow accelerators actually win.

*Your reasoning here (ungraded — for your own learning).*


---
## Summary

In this lab you:
1. **Calculated** arithmetic intensity from first principles for prefill and decode operators
2. **Constructed** an empirical roofline for your GPU using measured peaks
3. **Profiled** real LLM kernels and placed them on the roofline
4. **Diagnosed** performance bottlenecks and committed to one optimization
5. **Compared** GPU and Spyre using a dual roofline and the capacity threshold
6. **Exported** `lab6_results.json` for the Self-Assessment notebook

### Key Takeaways
- LLM decode is deeply memory-bound on all current GPUs (AI ~1 vs. ridge ~100–295)
- The same operation can be compute-bound or memory-bound depending on batch size
- Dataflow accelerators with on-chip scratchpad shift the roofline for workloads that *fit* — the crossover is a **capacity** threshold, not an AI value
- Quantization moves a memory-bound operator **along** the bandwidth roof; it does not raise the roof

---

### Wrap-up checklist
- [ ] **Runtime → Change runtime type → T4 GPU** set before running anything.
- [ ] Runtime → Run all; every **Checkpoint** prints `[OK]`.
- [ ] Parts 1–3 complete: the three AI values, your measured peaks, the profiled kernels.
- [ ] Part 4 classification table printed and both structured answers set.
- [ ] Part 5 dual roofline plotted and the capacity arithmetic filled in.
- [ ] Part 5b prefill/decode AI computed; Part 6 Landscape Map row complete.
- [ ] Part 7 run, `lab6_results.json` written and copied.
- [ ] JSON pasted into the grader notebook on Coursera, which reports your 40 ungraded Self-Assessment points.
- [ ] **Concept Check quiz** taken — a separate auto-graded Coursera Graded
      Assignment worth **10 graded points**. Mandatory, 80% to pass.

_End of Lab 6. This is the final lab of the course._

---

## Optional: Spyre Deep Dive

For students with access to IBM Spyre documentation or the `torch-spyre` package:
- Calculate the exact tiling strategy needed to fit one transformer layer's weights in the 64 MB scratchpad
- Determine how many layers can be pipelined across Spyre's 32 cores
- Estimate the latency improvement for autoregressive decode on a 125M parameter model

This section is **entirely optional and ungraded** — it earns no points and is not submitted.

---

*Course: Dataflow Architectures for AI, Week 6 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*


> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.
